# Golden Batch Early Warning System — Demo

1. Simulate batch trajectories with failure modes
2. Extract features (full and partial)
3. Build golden-batch PCA model
4. Score batches and check alerts
5. Train early release-risk model
6. Generate validation memo

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve() / '..' / 'src'))

from sim.generate_batch_trajectories import generate_batch_trajectories
from features.align_and_extract_features import extract_full_features, extract_partial_features, get_feature_columns
from monitoring.build_golden_batch_model import build_golden_batch_model
from monitoring.score_running_batches import score_cohort
from models.predict_release_risk import train_early_risk_model
from reports.model_validation_memo import generate_model_validation_memo

## 1. Simulate batch trajectories

In [2]:
traj, outcomes = generate_batch_trajectories(n_batches=500, seed=42)
print(f'Trajectories: {traj.shape}')
print(f'Outcomes: {outcomes.shape}')
print(f'Release pass rate: {outcomes["release_pass"].mean():.1%}')
print(f'Failure modes:\n{outcomes["failure_mode"].value_counts()}')

Trajectories: (20000, 12)
Outcomes: (500, 12)
Release pass rate: 71.8%
Failure modes:
failure_mode
none                   212
over_activation        161
delayed_harvest         87
nutrient_limitation     23
low_transduction        11
contamination_upset      6
Name: count, dtype: int64


## 2. Extract features

In [3]:
features = extract_full_features(traj)
fcols = get_feature_columns(features)
print(f'Full features: {features.shape}, {len(fcols)} numeric columns')

partial_d5 = extract_partial_features(traj, horizon_day=5.0)
print(f'Day-5 partial features: {partial_d5.shape}')
# Verify no future leakage
assert all('final' not in c for c in partial_d5.columns)

Full features: (500, 73), 72 numeric columns


Day-5 partial features: (500, 47)


## 3. Build golden-batch model

In [4]:
model = build_golden_batch_model(features, outcomes, fcols)
print(f'Training batches: {model["n_train"]}')
print(f'Components: {model["n_components"]}')
print(f'Variance explained: {model["total_variance_explained"]:.1%}')
print(f'T² limit: {model["t2_limit"]:.2f}')
print(f'SPE limit: {model["spe_limit"]:.2f}')

Training batches: 359
Components: 10
Variance explained: 93.4%
T² limit: 19.06
SPE limit: 10.46


## 4. Score all batches

In [5]:
scores = score_cohort(features, model)
print(f'Alert distribution:\n{scores["alert_state"].value_counts()}')

# Check healthy vs fault detection
good_ids = set(outcomes[outcomes['release_pass']]['batch_id'])
fault_ids = set(outcomes[outcomes['failure_mode'] != 'none']['batch_id'])
good_normal = scores[scores['batch_id'].isin(good_ids)]['alert_state'].eq('normal').mean()
fault_alert = scores[scores['batch_id'].isin(fault_ids)]['alert_state'].ne('normal').mean()
print(f'Good batch normal rate: {good_normal:.1%}')
print(f'Fault batch alert rate: {fault_alert:.1%}')

Alert distribution:
alert_state
normal         362
critical        93
t2_warning      24
spe_warning     21
Name: count, dtype: int64
Good batch normal rate: 90.0%
Fault batch alert rate: 45.1%


## 5. Early release-risk model (Day 5)

In [6]:
partial_fcols = get_feature_columns(partial_d5)
risk = train_early_risk_model(partial_d5, outcomes, partial_fcols)
print(f'CV AUC: {risk["metrics"]["cv_auc_mean"]}')
print(f'CV Brier: {risk["metrics"]["cv_brier_mean"]}')
print(f'Benchmark GBM AUC: {risk["metrics"]["benchmark_cv_auc_mean"]}')
risk['coefficients'].head(8)

CV AUC: 0.792
CV Brier: 0.1516
Benchmark GBM AUC: 0.809


,feature,coefficient
14,viable_cell_density_e6_ml_slope,1.5235
12,viable_cell_density_e6_ml_last,1.5186
11,viable_cell_density_e6_ml_std,1.5184
10,viable_cell_density_e6_ml_mean,1.3602
17,viability_pct_last,-0.9708
8,lactate_mm_initial,-0.9691
5,lactate_mm_mean,-0.9349
16,viability_pct_std,0.9238


## 6. Validation memo

In [7]:
html = generate_model_validation_memo(
    model, outcomes, risk,
    output_path=Path('..') / 'artifacts' / 'reports' / 'validation_memo.html'
)
assert 'Intended Use' in html
assert 'Limitations' in html
print(f'Memo generated: {len(html)} chars')
print('All checks passed.')

Memo generated: 3389 chars
All checks passed.
